In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import numpy as np
import json
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings

keras = tf.keras
layers = tf.keras.layers


warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
NPZ_PATH = r'.....'  # Path to folder containing preprocessed mfcc data, stored in .npz format
METADATA_PATH = r'.....'  # Path to metadata
BATCH_SIZE = 32
EPOCHS = 300
LEARNING_RATE = 0.001

np.random.seed(42)
tf.random.set_seed(42)

def configure_gpu_force():
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        print(f"Found {len(gpus)} GPU(s):")
        for gpu in gpus:
            print(f"  - {gpu}")
        
        
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("Memory growth enabled")
            return True
        except Exception as e:
            print(f"Could not set memory growth: {e}")
            return True
    else:
        print("No GPU found by TensorFlow")
        return False


class MFCC_CNN(tf.keras.Model):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Block 1: (Input)
        self.conv1 = keras.Sequential([
            layers.Conv2D(32, kernel_size=3, padding='same'), 
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(32, kernel_size=3, padding='same'), 
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D(2),
            layers.Dropout(0.1)
        ])
        # Block 2
        self.conv2 = keras.Sequential([
            layers.Conv2D(64, kernel_size=3, padding='same'), 
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(64, kernel_size=3, padding='same'), 
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D(2),
            layers.Dropout(0.2)
        ])
        #Block 3
        self.conv3 = keras.Sequential([
            layers.Conv2D(128, kernel_size=3, padding='same'), 
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(128, kernel_size=3, padding='same'),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D(2),
            layers.Dropout(0.3)
        ])
        #Block 4
        self.conv4 = keras.Sequential([
            layers.Conv2D(256, kernel_size=3, padding='same'),  
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.4)
        ])
        
        #
        self.fc = keras.Sequential([
            layers.Dense(128),  
            layers.ReLU(),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(64),   
            layers.ReLU(),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(num_classes)
        ])
    
    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x)
        return x


class DeadNeuronMonitor:
    def __init__(self, model, threshold=0.95):
        self.model = model
        self.threshold = threshold
        self.hooks = []
        self.activations = {}
        
    def check_dead_neurons(self, data):
        layer_outputs = []
        layer_names = []
        
        for layer in self.model.layers:
            if isinstance(layer, (layers.Conv2D, layers.Dense, layers.ReLU)):
                layer_outputs.append(layer.output)
                layer_names.append(layer.name)
        
        if len(layer_outputs) > 0:
            activation_model = keras.Model(inputs=self.model.input, outputs=layer_outputs)
            activations = activation_model.predict(data, verbose=0, batch_size=32)
            
            total_dead = 0
            total_neurons = 0
            
            for name, activation in zip(layer_names, activations):
                
                if len(activation.shape) == 4: 
                    batch, h, w, channels = activation.shape
                    activation = activation.reshape(batch, channels, -1)
                    inactive = (activation <= 0).mean(axis=2)
                    dead = (inactive > self.threshold).sum()
                    total_dead += dead
                    total_neurons += channels
                elif len(activation.shape) == 2: 
                    batch, features = activation.shape
                    inactive = (activation <= 0).mean(axis=0)
                    dead = (inactive > self.threshold).sum()
                    total_dead += dead
                    total_neurons += features
            
            if total_neurons > 0:
                return (total_dead / total_neurons) * 100
        return 0.0


def train_model(model, train_data, val_data, epochs, learning_rate=LEARNING_RATE):
    X_train, y_train = train_data
    X_val, y_val = val_data
    
   
    num_classes = model.fc.layers[-1].units
    y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_onehot = tf.keras.utils.to_categorical(y_val, num_classes)
    
    
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_onehot))
    train_dataset = train_dataset.shuffle(min(1000, len(X_train))).batch(BATCH_SIZE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val_onehot))
    val_dataset = val_dataset.batch(BATCH_SIZE)
    
   
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )
    
    
    criterion = tf.keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=0.1)
    
    
    train_loss_metric = tf.keras.metrics.Mean()
    train_acc_metric = tf.keras.metrics.CategoricalAccuracy()
    val_loss_metric = tf.keras.metrics.Mean()
    val_acc_metric = tf.keras.metrics.CategoricalAccuracy()
    
    
    monitor = DeadNeuronMonitor(model)
    
   
    history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'dead_train': [],
        'dead_val': []
    }
    
    best_val_acc = 0
    
    print("="*70)
    print("TRAINING ON GTX 1650 GPU")
    print("="*70)
    print(f"{'Epoch':<6} | {'Train Loss':<10} | {'Val Loss':<10} | {'Train Acc':<10} | {'Val Acc':<10} | {'Dead %':<10}")
    print("-" * 70)
    
    for epoch in range(epochs):
       
        model.trainable = True
        train_loss_metric.reset_state()
        train_acc_metric.reset_state()
        
        pbar = tqdm(train_dataset, desc=f'Epoch {epoch+1}')
        for batch_idx, (data, target) in enumerate(pbar):
            with tf.GradientTape() as tape:
                output = model(data, training=True)
                loss = criterion(target, output)
            
            gradients = tape.gradient(loss, model.trainable_variables)
            
           
            gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
            
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            
            
            train_loss_metric.update_state(loss)
            train_acc_metric.update_state(target, output)
            
            pbar.set_postfix({'Loss': train_loss_metric.result().numpy(), 
                             'Acc': train_acc_metric.result().numpy()})
        
        
        train_loss = train_loss_metric.result().numpy()
        train_acc = train_acc_metric.result().numpy()
        
        
        model.trainable = False
        val_loss_metric.reset_state()
        val_acc_metric.reset_state()
        
        for data, target in val_dataset:
            output = model(data, training=False)
            loss = criterion(target, output)
            
            val_loss_metric.update_state(loss)
            val_acc_metric.update_state(target, output)
        
        val_loss = val_loss_metric.result().numpy()
        val_acc = val_acc_metric.result().numpy()
        
        
        if epoch % 5 == 0:  
            sample_data = next(iter(train_dataset.take(1)))[0]
            dead_train = monitor.check_dead_neurons(sample_data[:16])  
            sample_val = next(iter(val_dataset.take(1)))[0]
            dead_val = monitor.check_dead_neurons(sample_val[:16])
        else:
            dead_train = history['dead_train'][-1] if history['dead_train'] else 0
            dead_val = history['dead_val'][-1] if history['dead_val'] else 0
        
        
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['dead_train'].append(dead_train)
        history['dead_val'].append(dead_val)
        
        
        print(f"{epoch+1:<6} | {train_loss:<10.4f} | {val_loss:<10.4f} | "
              f"{train_acc:<10.4f} | {val_acc:<10.4f} | {dead_train:<10.1f}")
        
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model.save_weights('best_model.weights.h5')
            checkpoint_data = {
                'epoch': epoch,
                'val_acc': val_acc,
                'history': history
            }
            np.save('best_model_checkpoint.npy', checkpoint_data, allow_pickle=True)
        
    
    return model, history


def results_table(history):

    print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Val Loss':<12} | "
          f"{'Train Acc':<12} | {'Val Acc':<12} | {'Dead %':<12}")
    print("-" * 90)
    
    
    for i in range(0, len(history['epoch']), 10):
        if i < len(history['epoch']):
            epoch = history['epoch'][i]
            train_loss = history['train_loss'][i]
            val_loss = history['val_loss'][i]
            train_acc = history['train_acc'][i]
            val_acc = history['val_acc'][i]
            dead_pct = history['dead_train'][i]
            
            print(f"{epoch:<6} | {train_loss:<12.4f} | {val_loss:<12.4f} | "
                  f"{train_acc:<12.4f} | {val_acc:<12.4f} | {dead_pct:<12.1f}")
    
   
    if len(history['epoch']) > 0:
        i = -1
        epoch = history['epoch'][i]
        train_loss = history['train_loss'][i]
        val_loss = history['val_loss'][i]
        train_acc = history['train_acc'][i]
        val_acc = history['val_acc'][i]
        dead_pct = history['dead_train'][i]
        
        print("-" * 90)
        print(f"{epoch:<6} | {train_loss:<12.4f} | {val_loss:<12.4f} | "
              f"{train_acc:<12.4f} | {val_acc:<12.4f} | {dead_pct:<12.1f}")

def training_curves(history):
  
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    
    axes[0].plot(history['epoch'], history['train_acc'], label='Train', marker='o', markersize=3)
    axes[0].plot(history['epoch'], history['val_acc'], label='Val', marker='s', markersize=3)
    axes[0].axhline(y=0.96, color='r', linestyle='--', label='Target 96%')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Accuracy Progress')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    
    axes[1].plot(history['epoch'], history['train_loss'], label='Train', marker='o', markersize=3)
    axes[1].plot(history['epoch'], history['val_loss'], label='Val', marker='s', markersize=3)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss Progress')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    
    axes[2].plot(history['epoch'], history['dead_train'], label='Train', marker='o', markersize=3)
    axes[2].plot(history['epoch'], history['dead_val'], label='Val', marker='s', markersize=3)
    axes[2].axhline(y=30, color='r', linestyle='--', label='Warning (30%)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Dead Neurons (%)')
    axes[2].set_title('Dead Neuron Monitoring')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def load_data():
    
    print(f"Looking for data at: {NPZ_PATH}")
    
    if not os.path.exists(NPZ_PATH):
        print(f"Data file not found at {NPZ_PATH}")
        return None, None, None, None
    
   
    data = np.load(NPZ_PATH, allow_pickle=True)
    
    
    X_train = data['X_train']
    y_train = data['y_train']
    X_val = data['X_val']
    y_val = data['y_val']
    
    print(f"Data loaded from npz:")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  y_train shape: {y_train.shape}")
    print(f"  X_val shape: {X_val.shape}")
    print(f"  y_val shape: {y_val.shape}")
    
    
    if os.path.exists(METADATA_PATH):
        with open(METADATA_PATH, 'r') as f:
            metadata = json.load(f)
        print(f"  Number of classes: {len(metadata['class_names'])}")
        print(f"  Class names: {metadata['class_names']}")
    
    data.close()
    return X_train, y_train, X_val, y_val


def main():
    
    print("\nAll physical devices:")
    all_devices = tf.config.list_physical_devices()
    for device in all_devices:
        print(f"  - {device}")
    
    gpu_available = configure_gpu_force()

    print("\nLoading data from npz file...")
    data = load_data()
    
    if data[0] is None:
        print("Failed to load data. Exiting.")
        return
    
    X_train, y_train, X_val, y_val = data
    print(f"\nData shapes:")
    print(f"X_train: {X_train.shape}")
    print(f"y_train: {y_train.shape}")
    
    
    print("\n Normalizing data...")
    mean = np.mean(X_train, axis=(0, 1, 2), keepdims=True)
    std = np.std(X_train, axis=(0, 1, 2), keepdims=True) + 1e-8
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    
    
    X_train = np.clip(X_train, -3, 3)
    X_val = np.clip(X_val, -3, 3)
    
    num_classes = len(np.unique(y_train))
    model = MFCC_CNN(num_classes=num_classes)
    
    model = MFCC_CNN(num_classes=num_classes)
    
    
    print("\n" + "="*70)
    model, history = train_model(
        model,
        (X_train, y_train),
        (X_val, y_val),
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE
    )
    
    
    results_table(history)
    training_curves(history)

    
    
    if os.path.exists('best_model.weights.h5'):
        model.load_weights('best_model.weights.h5')
        print("Loaded best model weights")
    
    
    num_classes = model.fc.layers[-1].units
    y_val_onehot = tf.keras.utils.to_categorical(y_val, num_classes)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val_onehot))
    val_dataset = val_dataset.batch(BATCH_SIZE)
    
    
    val_acc_metric = tf.keras.metrics.CategoricalAccuracy()
    
    for data, target in val_dataset:
        output = model(data, training=False)
        val_acc_metric.update_state(target, output)
    
    final_acc = val_acc_metric.result().numpy()
    
    
    if os.path.exists('checkpoint.npy'):
        checkpoint = np.load('checkpoint.npy', allow_pickle=True).item()
        best_val_acc = checkpoint['val_acc']
        print(f"Best validation accuracy: {best_val_acc:.4f}")
    
    print(f"Final validation accuracy: {final_acc:.4f}")
    
    
    model.save_weights('model.weights.h5')
    
    
    
    np.savez('normalization_params.npz', mean=mean, std=std)
    print(" Normalization params saved to: normalization_params.npz")
    
  
    print(f"Final Validation Accuracy: {final_acc:.4f}")
    print(f"Epochs Trained: {len(history['epoch'])}")


if __name__ == "__main__":
    tf.config.optimizer.set_jit(True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16') 
    main()